In [1]:
import os
import pandas as pd

# 현재 폴더 내의 모든 CSV 파일을 읽어서 하나의 DataFrame으로 적재
csv_files = [f for f in os.listdir('.') if f.endswith('.csv')]
if not csv_files:
    raise ValueError("현재 폴더에 CSV 파일이 없습니다.")

df_list = []
for file in csv_files:
    temp_df = pd.read_csv(file)
    df_list.append(temp_df)

df = pd.concat(df_list, ignore_index=True)
print(f"총 {len(df)}개의 행이 적재되었습니다.")


총 499개의 행이 적재되었습니다.


In [2]:
df.columns

Index(['time', 'pods', 'cpu', 'memory', 'karpenter', 'kubecaps', 'region'], dtype='object')

In [4]:
import ast
def to_dict(x):
    if isinstance(x, dict):
        return x
    try:
        return ast.literal_eval(x)
    except Exception:
        return {}

df['kubecaps'] = df['kubecaps'].apply(to_dict)

df['nodepool'] = df['kubecaps'].apply(lambda x: x.get('nodepool', None))
df['nodepool_instances'] = df['nodepool'].apply(
    lambda lst: sum(item.get('num_instances', 0) for item in lst)
)

df = df.sort_values(by="nodepool_instances", ascending=False, ignore_index=True)
df[['pods', 'cpu', 'memory', 'nodepool_instances', 'nodepool']].to_csv("d.csv")
